In [2]:
# Import Libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    StackingRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Summary
# Load the preprocessed dataset and
# split it into training and validation sets.

X = pd.read_csv(
    "../Outputs/pipeline1_train_preprocessed.csv"
)

y = np.log1p(
    pd.read_csv(
        "../Outputs/pipeline1_train_target.csv"
    )["SalePrice"]
)

print("Dataset loaded successfully!\n")

print(f"Dataset Shape : {X.shape}")

print(f"Missing Values: {X.isnull().sum().sum()}")

Dataset loaded successfully!

Dataset Shape : (1460, 208)
Missing Values: 1468


In [4]:
X_train, X_test, y_train, y_test = train_test_split(

    X.fillna(0),

    y,

    test_size=0.2,

    random_state=42

)

# Create 5-Fold Cross Validation

kf = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

print("Train/Test Split completed!\n")

print(f"Train Shape : {X_train.shape}")

print(f"Test Shape  : {X_test.shape}")

Train/Test Split completed!

Train Shape : (1168, 208)
Test Shape  : (292, 208)


In [5]:
# RMSE Cross Validation Function

def rmse_cv(

    model,

    X,

    y

):

    scores = cross_val_score(

        model,

        X,

        y,

        scoring="neg_root_mean_squared_error",

        cv=kf,

        n_jobs=-1

    )

    return -scores

In [6]:
# I Create tuned machine learning models.

xgb = XGBRegressor(

    colsample_bytree=0.8,

    learning_rate=0.03,

    max_depth=3,

    n_estimators=1000,

    subsample=0.8,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1,

    verbosity=0

)

lgbm = LGBMRegressor(

    learning_rate=0.03,

    max_depth=3,

    n_estimators=1000,

    num_leaves=15,

    subsample=0.7,

    random_state=42,

    n_jobs=-1,

    verbosity=-1

)

rf = RandomForestRegressor(

    n_estimators=700,

    max_depth=20,

    max_features="sqrt",

    min_samples_leaf=1,

    min_samples_split=2,

    random_state=42,

    n_jobs=-1

)

print("Models created successfully!")

xgb.fit(
    X_train,
    y_train
)

lgbm.fit(
    X_train,
    y_train
)

rf.fit(
    X_train,
    y_train
)

print("All models trained successfully!")

Models created successfully!
All models trained successfully!


In [7]:
# Cross Validation Comparison

results = {}

models = {

    "XGBoost": xgb,

    "LightGBM": lgbm,

    "Random Forest": rf

}

for name, model in models.items():
    

    scores = rmse_cv(

    model,

    X_train,

    y_train

)

    results[name] = [

        scores.mean(),

        scores.std()

    ]

results = pd.DataFrame(

    results,

    index=[

        "Mean RMSE",

        "Std RMSE"

    ]

).T

results = results.sort_values(

    by="Mean RMSE"

)

display(results)

,Mean RMSE,Std RMSE
XGBoost,0.119914,0.016041
LightGBM,0.125296,0.013385
Random Forest,0.136283,0.015588


# Weighted Averaging Ensemble start right now

#Tried weighted averaging ensemble, but it did not outperform the tuned XGBoost model.

In [8]:
# Weighted Averaging Ensemble

xgb_pred = xgb.predict(X_test)

lgbm_pred = lgbm.predict(X_test)

rf_pred = rf.predict(X_test)

ensemble_pred = (

    0.7 * xgb_pred +

    0.2 * lgbm_pred +

    0.1 * rf_pred

)

print("Weighted Ensemble predictions completed!")

Weighted Ensemble predictions completed!


In [9]:
# Evaluate Weighted Ensemble

rmse = np.sqrt(

    mean_squared_error(

        y_test,

        ensemble_pred

    )

)

r2 = r2_score(

    y_test,

    ensemble_pred

)

print("=" * 60)

print("Weighted Averaging Ensemble")

print("=" * 60)

print(f"RMSE : {rmse:.5f}")

print(f"R²   : {r2:.5f}")

Weighted Averaging Ensemble
RMSE : 0.13232
R²   : 0.90617


In [10]:
# Compare Different Ensemble Weights

weights = {

    "80-20 (XGB+LGBM)": [0.8, 0.2, 0.0],

    "70-30 (XGB+LGBM)": [0.7, 0.3, 0.0],

    "60-40 (XGB+LGBM)": [0.6, 0.4, 0.0],

    "70-20-10": [0.7, 0.2, 0.1],

    "60-20-20": [0.6, 0.2, 0.2]

}

results = []

for name, (wx, wl, wr) in weights.items():

    pred = (

        wx * xgb.predict(X_test)

        + wl * lgbm.predict(X_test)

        + wr * rf.predict(X_test)

    )

    rmse = np.sqrt(

        mean_squared_error(

            y_test,

            pred

        )

    )

    r2 = r2_score(

        y_test,

        pred

    )

    results.append([

        name,

        rmse,

        r2

    ])

results = pd.DataFrame(

    results,

    columns=[

        "Ensemble",

        "RMSE",

        "R²"

    ]

)

results = results.sort_values(

    by="RMSE"

).reset_index(drop=True)

display(results)

,Ensemble,RMSE,R²
0,80-20 (XGB+LGBM),0.132233,0.906300
1,70-20-10,0.132321,0.906174
2,70-30 (XGB+LGBM),0.132718,0.905610
3,60-20-20,0.132770,0.905537
4,60-40 (XGB+LGBM),0.133289,0.904797


#Stacking ensemble is here

In [11]:
# Create Stacking Ensemble

stack_model = StackingRegressor(

    estimators=[

        ("xgb", xgb),

        ("lgbm", lgbm),

        ("rf", rf)

    ],

    final_estimator=LinearRegression(),

    cv=5,

    n_jobs=-1,

    passthrough=False

)

print("Stacking model created successfully!")

Stacking model created successfully!


In [12]:
# Train Stacking Ensemble

stack_model.fit(

    X_train,

    y_train

)

print("Stacking model trained successfully!")

Stacking model trained successfully!


In [13]:
# Evaluate Stacking Ensemble

stack_pred = stack_model.predict(

    X_test

)

rmse = np.sqrt(

    mean_squared_error(

        y_test,

        stack_pred

    )

)

r2 = r2_score(

    y_test,

    stack_pred

)

print("=" * 60)

print("Stacking Ensemble")

print("=" * 60)

print(f"RMSE : {rmse:.5f}")

print(f"R²   : {r2:.5f}")

Stacking Ensemble
RMSE : 0.13219
R²   : 0.90637


In [14]:
# Cross Validation

scores = rmse_cv(

    stack_model,

    X.fillna(0),

    y

)

print("=" * 60)

print("Stacking Cross Validation")

print("=" * 60)

print(f"Mean RMSE : {scores.mean():.5f}")

print(f"Std RMSE  : {scores.std():.5f}")

Stacking Cross Validation
Mean RMSE : 0.12550
Std RMSE  : 0.01831
